# Project: Road Traffic Accidents Analysis in Finland
## Framework: CRISP-DM Process Model

### 1. Business Understanding
In this phase, we explore the official spatial dataset of road traffic accidents provided by Statistics Finland (Tilastokeskus). Our objective is to determine if environmental and seasonal features (such as winter months) act as core triggers for highly severe road accidents, and to analyze this relationship using spatial visualization and Linear Regression modeling.

In [29]:
# TODO: tämä selitys vanhentunut ei käytetä lin. regressiota

### 2. Data Understanding
In this phase, we explore the official spatial dataset of road traffic accidents provided by Statistics Finland (Tilastokeskus). Our objective is to determine if environmental and seasonal features (such as winter months) act as core triggers for highly severe road accidents, and to analyze this relationship using spatial visualization and Linear Regression modeling.

In [30]:
# 1. Install and verify required spatial and ML libraries
%pip install geopandas contextily requests fmiopendata mlxtend scikit-learn

# 2. Import core engines
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import contextily as ctx
import datetime as dt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from fmiopendata.wfs import download_stored_query
print("All core engines and machine learning libraries imported successfully!")

Note: you may need to restart the kernel to use updated packages.
All core engines and machine learning libraries imported successfully!



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\Local Admin\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [31]:
# 3. Data Loading - Correcting the separator to comma ',' as per actual data structure
csv_path = 'datasets/tieliikenne_2024.csv'
df_accidents = pd.read_csv(csv_path, sep=',')


In [32]:
print("Shape (rows, columns):", df_accidents.shape)
print("\nColumns:", list(df_accidents.columns))
print("\nData types:")
print(df_accidents.dtypes)

Shape (rows, columns): (2809, 17)

Columns: ['FID', 'id', 'geom', 'vvonn', 'kkonn', 'kello', 'vakav', 'onntyyppi', 'lkmhapa', 'lkmlaka', 'lkmjk', 'lkmpp', 'lkmmo', 'lkmmp', 'lkmmuukulk', 'x', 'y']

Data types:
FID               str
id              int64
geom              str
vvonn           int64
kkonn           int64
kello             str
vakav           int64
onntyyppi       int64
lkmhapa         int64
lkmlaka         int64
lkmjk           int64
lkmpp           int64
lkmmo           int64
lkmmp           int64
lkmmuukulk      int64
x             float64
y             float64
dtype: object


In total, there are 17 columns of data in the dataset. 

In [33]:
print("\nMissing values per column:")
print(df_accidents.isna().sum())


Missing values per column:
FID           0
id            0
geom          0
vvonn         0
kkonn         0
kello         1
vakav         0
onntyyppi     0
lkmhapa       0
lkmlaka       0
lkmjk         0
lkmpp         0
lkmmo         0
lkmmp         0
lkmmuukulk    0
x             0
y             0
dtype: int64


There is only one missing data for **kello**.

### 3. Data Preparation
Since raw accident records represent single events, direct continuous linear modeling on row levels is mathematically invalid. We perform an **Aggregation step** to compute total accidents grouped by month, establishing a true continuous numeric target variable suitable for Linear Regression modeling.

In [34]:
# Readable column names for presentation
readable_col_names =  {
"vvonn" : "vuosi",
"kkonn" : "kuukausi",
"kello" : "kello",
"vakav" : "vakavuus",
"onntyyppi" : "onnettomuustyyppi",
"lkmhapa" : "henkilö- ja pakettiautot",
"lkmlaka" : "linja- ja kuorma-autot",
"lkmjk" : "jalankulkijat",
"lkmpp" : "polkupyörät ja kevyet sähköajoneuvot",
"lkmmo" : "mopot",
"lkmmp" : "moottoripyörät",
"lkmmuukulk" : "muut kulkuneuvot",
"x" : "x",
"y" : "y"
}
#
df_accidents.rename( columns=readable_col_names, inplace=True)

In [35]:
#  Luokkamuuttujat:
'''
vakav = onnettomuuden vakavuus: 1 = kuolemaan johtanut onnettomuus, 2 = loukkaantumiseen johtanut onnettomuus, 3 = vakavaan loukkaantumiseen johtanut onnettomuus

onntyyppi = onnettomuustyyppi: 0 = samat ajosuunnat (ajo suoraan), 1 = samat ajosuunnat (ajo kääntyen), 2 = vastakkaiset ajosuunnat (ajo suoraan), 3 = vastakkaiset ajosuunnat (ajo kääntyen), 4 = risteävät ajosuunnat (ajo suoraan), 5 = risteävät ajosuunnat (ajo kääntyen), 6 = jalankulkijaonnettomuus (suojatiellä), 7 = jalankulkijaonnettomuus (muualla), 8 = tieltä suistuminen, 9 = muu onnettomuus
'''

'\nvakav = onnettomuuden vakavuus: 1 = kuolemaan johtanut onnettomuus, 2 = loukkaantumiseen johtanut onnettomuus, 3 = vakavaan loukkaantumiseen johtanut onnettomuus\n\nonntyyppi = onnettomuustyyppi: 0 = samat ajosuunnat (ajo suoraan), 1 = samat ajosuunnat (ajo kääntyen), 2 = vastakkaiset ajosuunnat (ajo suoraan), 3 = vastakkaiset ajosuunnat (ajo kääntyen), 4 = risteävät ajosuunnat (ajo suoraan), 5 = risteävät ajosuunnat (ajo kääntyen), 6 = jalankulkijaonnettomuus (suojatiellä), 7 = jalankulkijaonnettomuus (muualla), 8 = tieltä suistuminen, 9 = muu onnettomuus\n'

### Converting class variables to 1-hot encoded columns

In [42]:
# TODO: onnettomuustyyppi muutettava 1-hot sarakkeiksi
accident_types_raw = {
0 : "samat ajosuunnat (ajo suoraan)",
1 : "samat ajosuunnat (ajo kääntyen)",
2 : "vastakkaiset ajosuunnat (ajo suoraan)",
3 : "vastakkaiset ajosuunnat (ajo kääntyen)",
4 : "risteävät ajosuunnat (ajo suoraan)",
5 : "risteävät ajosuunnat (ajo kääntyen)",
6 : "jalankulkijaonnettomuus (suojatiellä)",
7 : "jalankulkijaonnettomuus (muualla)",
8 : "tieltä suistuminen",
9 : "muu onnettomuus"
}

accident_types_columns = {
0 : "type0",
1 : "type1",
2 : "type2",
3 : "type3",
4 : "type4",
5 : "type5",
6 : "type6",
7 : "type7",
8 : "type8",
9 : "type9"
}
# Create columns 'typeX' and assign boolean 1 or 0
for type_number, column_name in accident_types_columns.items():
    df_accidents[column_name] = [1 if accident_type == type_number else 0 for accident_type in df_accidents['onnettomuustyyppi']]

df_accidents.head(1)


,FID,id,geom,vuosi,kuukausi,kello,vakavuus,onnettomuustyyppi,henkilö- ja pakettiautot,linja- ja kuorma-autot,...,type1,type2,type3,type4,type5,type6,type7,type8,type9,tunti
0,tieliikenne_2024.1,1,POINT (423323.0412999997 6695927.4465),2024,7,00.00-00.59,2,3,1,0,...,0,0,1,0,0,0,0,0,0,0


In [43]:
# Convert accident lethality to 0 or 1
df_accidents['vakavuus'] = [1 if vakavuus == 1 else 0 for vakavuus in df_accidents['vakavuus']]

,FID,id,geom,vuosi,kuukausi,kello,vakavuus,onnettomuustyyppi,henkilö- ja pakettiautot,linja- ja kuorma-autot,...,type1,type2,type3,type4,type5,type6,type7,type8,type9,tunti
0,tieliikenne_2024.1,1,POINT (423323.0412999997 6695927.4465),2024,7,00.00-00.59,0,3,1,0,...,0,0,1,0,0,0,0,0,0,0
1,tieliikenne_2024.2,2,POINT (306376.20660000015 6761653.649800001),2024,10,00.00-00.59,0,8,1,0,...,0,0,0,0,0,0,0,1,0,0
2,tieliikenne_2024.3,3,POINT (259784.6640999997 6994822.009199999),2024,11,22.00-22.59,0,1,2,0,...,1,0,0,0,0,0,0,0,0,22
3,tieliikenne_2024.4,4,POINT (392858.12179999985 6681502.598200001),2024,1,15.00-15.59,0,0,1,1,...,0,0,0,0,0,0,0,0,0,15
4,tieliikenne_2024.5,5,POINT (383665.0440999996 6671405.377699999),2024,1,17.00-17.59,0,8,1,0,...,0,0,0,0,0,0,0,1,0,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2804,tieliikenne_2024.2805,2805,POINT (126484.77840000018 6682647.3047),2024,11,12.00-12.59,0,8,0,1,...,0,0,0,0,0,0,0,1,0,12
2805,tieliikenne_2024.2806,2806,POINT (107800.28100000042 6683502.192500001),2024,12,17.00-17.59,0,8,1,0,...,0,0,0,0,0,0,0,1,0,17
2806,tieliikenne_2024.2807,2807,POINT (116624.37719999999 6701750.947699999),2024,12,15.00-15.59,0,1,3,0,...,1,0,0,0,0,0,0,0,0,15
2807,tieliikenne_2024.2808,2808,POINT (110664.54220000003 6684589.5315000005),2024,12,08.00-08.59,0,0,2,0,...,0,0,0,0,0,0,0,0,0,8


### Converting cyclical time variable to linear difference from midday

In [37]:
# TODO: kellonaika muutettava poikkeamaksi kello 12:00 siten, että ajan arvo on välillä 0-12

In [38]:
# Spatial Transformation - Convert to GeoDataFrame using Finnish coordinate system (EPSG:3067)
accidents_gdf = gpd.GeoDataFrame(
    df_accidents,
    geometry=gpd.points_from_xy(df_accidents.x, df_accidents.y),
    crs="EPSG:3067"
)

print(" Data loaded and transformed successfully into Spatial GeoDataFrame!")
display(accidents_gdf.head())

 Data loaded and transformed successfully into Spatial GeoDataFrame!


,FID,id,geom,vuosi,kuukausi,kello,vakavuus,onnettomuustyyppi,henkilö- ja pakettiautot,linja- ja kuorma-autot,...,type1,type2,type3,type4,type5,type6,type7,type8,type9,geometry
0,tieliikenne_2024.1,1,POINT (423323.0412999997 6695927.4465),2024,7,00.00-00.59,2,3,1,0,...,0,0,1,0,0,0,0,0,0,POINT (423323.041 6695927.446)
1,tieliikenne_2024.2,2,POINT (306376.20660000015 6761653.649800001),2024,10,00.00-00.59,2,8,1,0,...,0,0,0,0,0,0,0,1,0,POINT (306376.207 6761653.65)
2,tieliikenne_2024.3,3,POINT (259784.6640999997 6994822.009199999),2024,11,22.00-22.59,2,1,2,0,...,1,0,0,0,0,0,0,0,0,POINT (259784.664 6994822.009)
3,tieliikenne_2024.4,4,POINT (392858.12179999985 6681502.598200001),2024,1,15.00-15.59,2,0,1,1,...,0,0,0,0,0,0,0,0,0,POINT (392858.122 6681502.598)
4,tieliikenne_2024.5,5,POINT (383665.0440999996 6671405.377699999),2024,1,17.00-17.59,2,8,1,0,...,0,0,0,0,0,0,0,1,0,POINT (383665.044 6671405.378)


## 3.X Feature extraction

The hours are are in String datatype and in format XX.00-XX.59.

We take the first two characters of each string in column 'kello' and convert that to float. We convert it into float as there is a one missing value **(NaN)** that cannot be an Integer.

After dropping the missing row, conversion to whole numbers is done with **.astype(int)**

In [39]:
df_accidents['tunti'] = df_accidents['kello'].str[:2].astype(float)

# drop the row(s) with missing time, then make it an integer
df_accidents = df_accidents.dropna(subset=['tunti'])
df_accidents['tunti'] = df_accidents['tunti'].astype(int)

## 3.X functions

In [40]:
## Function to calculate how many hours is the given time from noon
def hour_from_noon(hour):
    return abs(12 - hour)

# print(hour_from_noon(12))
#print(hour_from_noon(24))
#print(hour_from_noon(1))
#print(hour_from_noon(4))


### 4. Modeling
Since raw accident records represent single events, direct continuous linear modeling on row levels is mathematically invalid. We perform an **Aggregation step** to compute total accidents grouped by month, establishing a true continuous numeric target variable suitable for Linear Regression modeling.

In [41]:
# 5. Data Preparation: Aggregate total accidents by month (kkonn)
monthly_df = accidents_gdf.groupby('kkonn').size().reset_index(name='accidents_count')

# 6. Define Linear Regression variables
X = monthly_df[['kkonn']]          # Explanatory variable (Month index)
y = monthly_df['accidents_count']  # Continuous numeric target

# 7. Train the Linear Regression Engine
reg_model = LinearRegression()
reg_model.fit(X, y)

# 8. Phase 5: Evaluation - Calculate MAE to measure average prediction error
y_pred = reg_model.predict(X)
mae_error = mean_absolute_error(y, y_pred)

print(" --- Linear Regression Evaluation Model Results ---")
print(f"Model Intercept (b0 - Base Baseline): {reg_model.intercept_:.2f}")
print(f"Model Coefficient (b1 - Slope Weighted): {reg_model.coef_[0]:.2f}")
print(f"Mean Absolute Error (MAE): {mae_error:.2f} accidents")

# 9. Visualize the Linear Fit and distribution
plt.figure(figsize=(10, 5))
sns.barplot(data=monthly_df, x='kkonn', y='accidents_count', color='skyblue', alpha=0.7, label='Actual Monthly Count')
plt.plot(monthly_df['kkonn'] - 1, y_pred, color='red', marker='o', linewidth=2, label='Linear Regression Fit')
plt.title('Road Traffic Accidents Linear Model Fit by Month', fontsize=12, fontweight='bold')
plt.xlabel('Month of the Year')
plt.ylabel('Total Accidents')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

KeyError: 'kkonn'

### 4.x FMI Weather Interface Testing
We test the FMI Open Data live streaming engine to collect auxiliary meteorological variables corresponding to the accident location spaces.

In [ ]:
import pandas as pd
import datetime as dt
from fmiopendata.wfs import download_stored_query

try:
    print("Connecting to FMI Government server... Fetching raw spatial coordinates...")
    
    # 1. Fetch data from FMI using a defined bounding box (Standard multipoint query)
    obs = download_stored_query(
        "fmi::observations::weather::multipointcoverage", 
        args=["bbox=24.5,60.1,25.1,60.3"]
    )
    
    # 2. Manual dictionary parsing loop to completely bypass 'to_dataframe' error
    weather_records = []
    
    # Loop through each unique timestamp key in the raw data
    for timestamp in obs.data.keys():
        # Loop through each weather station name key at this timestamp
        for station_name in obs.data[timestamp].keys():
            station_data = obs.data[timestamp][station_name]
            
            # Extract main target parameters provided by FMI
            temp = station_data.get('Air temperature', {}).get('value', None)
            wind = station_data.get('Wind speed', {}).get('value', None)
            rh = station_data.get('Relative humidity', {}).get('value', None)
            
            # Append rows manually to our collection list
            weather_records.append({
                "time": timestamp,
                "location": station_name,
                "air_temperature": temp,
                "wind_speed": wind,
                "relative_humidity": rh
            })
            
    # 3. Construct the clean Pandas DataFrame manually!
    weather_df = pd.DataFrame(weather_records)
    
    print("Real FMI Weather database constructed successfully without errors!")
    print(f"Weather Data Shape: {weather_df.shape}")
    
    # Display the first 10 rows for data understanding validation
    display(weather_df.head(10))

except Exception as e:
    print(f"Final Engine Error Notice: {e}. Check server status.")


### 5. Evaluation

### 6. Deployment